# Build Your Own RAG System (Workshop Notebook)

**Goal:** By the end, you will have a working Retrieval-Augmented Generation (RAG) system that answers questions from documents.

## How to use this notebook
Run cells **top to bottom**, one section at a time. Read the markdown first, then run the code.

## What you will learn
1. What embeddings are
2. How to load and chunk documents
3. How to store chunks in a vector database
4. How to retrieve relevant context and ask an LLM to answer


## Section 1: Setup

Before we start:
- Python 3.10+
- Dependencies installed: `pip install -r requirements.txt`
- API key in `.env`: `GEMINI_API_KEY=...`

**Facilitator note:** This section should take about 5 minutes.


In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / ".env")

api_key = os.getenv("GEMINI_API_KEY")
if not api_key or api_key == "your_key_here":
    raise ValueError("Add GEMINI_API_KEY to your .env file first.")

print("Setup OK")
print("Repo root:", ROOT)


## Section 2: What Are Embeddings?

An **embedding** turns text into a list of numbers that capture meaning.

- Similar meanings -> similar numbers
- This is how search works in modern AI

We will compare three sentences and see which pair is most similar.


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
import numpy as np

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

sentences = [
    "The student passed the examination",
    "The learner succeeded in the test",
    "The banana is yellow",
]

vectors = embeddings.embed_documents(sentences)


def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


print(
    "Similarity (exam sentence vs test sentence):",
    round(cosine_similarity(vectors[0], vectors[1]), 3),
)
print(
    "Similarity (exam sentence vs banana sentence):",
    round(cosine_similarity(vectors[0], vectors[2]), 3),
)


## Section 3: Load Documents

RAG starts with your source documents.

We will load the sample files in `sample_docs/`.

**Try this after the demo:** swap in your own `.txt` or `.pdf` file.


### Hint (Section 3)

Work through these steps before you code:

1. Open `rag_pipeline.py` in the repo root and find the function that **loads files from a folder**.
2. That function returns a **list of Document objects** (one per file or page).
3. Pass it the path to our sample folder: combine `ROOT` with `"sample_docs"`.
4. Assign the result to a variable named `documents`.

**Check yourself:** after running, `len(documents)` should be at least 3, and `documents[0].page_content` should contain readable text.


In [ ]:
# YOUR CODE HERE
# Hint: use rag_pipeline.load_documents(ROOT / "sample_docs")

from rag_pipeline import load_documents

documents = None  # replace this

print("Loaded documents:", len(documents))
print("First document preview:
")
print(documents[0].page_content[:300])


## Section 4: Chunk the Documents

LLMs cannot read a whole book at once. We split documents into smaller **chunks**.

Good defaults for this workshop:
- `chunk_size=500`
- `chunk_overlap=50`


### Hint (Section 4)

You already have `documents` from the previous step.

1. Find the **splitting** function in `rag_pipeline.py`.
2. It takes your document list and returns smaller **chunks**.
3. Default workshop settings are already built in (`chunk_size=500`, `chunk_overlap=50`) — you only need to pass `documents`.
4. Assign the result to `chunks`.

**Check yourself:** you should have **more chunks than documents**. If counts are equal, re-read the chunking section above.


In [ ]:
# YOUR CODE HERE
# Hint: use rag_pipeline.split_documents(documents)

from rag_pipeline import split_documents

chunks = None  # replace this

print("Number of chunks:", len(chunks))
print("Sample chunk:
")
print(chunks[0].page_content[:300])


## Section 5: Build the Vector Store (ChromaDB)

Now we:
1. Convert each chunk to an embedding
2. Save embeddings in a local database called **ChromaDB**

This runs on your laptop. No cloud database needed.


### Hint (Section 5)

This step connects chunking to storage:

1. Find the function that **builds a vector store** in `rag_pipeline.py`.
2. It needs two things: your `chunks` and a **folder path** where ChromaDB will save data locally.
3. Use `ROOT / "chroma_db"` as the save location (this creates a folder on your laptop).
4. Assign the returned vector store object to `vector_store`.

**Check yourself:** after running, you should see a `chroma_db/` folder appear under the repo root.


In [ ]:
# YOUR CODE HERE
# Hint: use rag_pipeline.build_vector_store(chunks, ROOT / "chroma_db")

from rag_pipeline import build_vector_store

vector_store = None  # replace this

print("Vector store created at:", ROOT / "chroma_db")


## Section 6: Retrieve Relevant Chunks

When a user asks a question, RAG searches the vector store for the most relevant chunks.

Run a test question and inspect what comes back.


### Hint (Section 6)

No new imports needed — use the `vector_store` object from Section 5.

1. Look for a **similarity search** method on the vector store object.
2. Pass the `question` string already defined in the cell.
3. Set `k=3` to retrieve the top 3 most relevant chunks.
4. Assign the result to `retrieved_docs`.

**Check yourself:** each item in `retrieved_docs` should have `.page_content` and `.metadata["source"]`.


In [ ]:
# YOUR CODE HERE
question = "What is the primary purpose of government in Nigeria?"

retrieved_docs = (
    None  # replace this using vector_store.similarity_search(question, k=3)
)

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"--- Chunk {i} | Source: {doc.metadata.get('source')} ---")
    print(doc.page_content[:250])
    print()


## Section 7: Build the RAG Answer

Now we combine:
- Retrieved context
- User question
- LLM generation

This is the core RAG pattern used in ChatGPT plugins, Perplexity, and many enterprise tools.


### Hint (Section 7)

This is the full RAG step — retrieval plus generation:

1. Import `answer_question` from `rag_pipeline`.
2. Call it with `vector_store` and the same `question` from Section 6.
3. It returns a **dictionary** with keys like `"answer"` and `"sources"`.
4. Assign that dictionary to `result`.

**Check yourself:** `result["answer"]` should mention government or welfare if retrieval worked. `result["sources"]` should list filenames.


In [ ]:
# YOUR CODE HERE
from rag_pipeline import answer_question

result = None  # replace this

print("Question:", question)
print("
Answer:
", result["answer"])
print("
Sources:", result["sources"])


## Section 8: Compare With and Without RAG

Ask the LLM directly without context. Then compare with your RAG answer.

This shows why retrieval matters.


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", google_api_key=api_key, temperature=0.2)

plain_answer = llm.invoke(question)
print("Without RAG:
", plain_answer.content)
print("
With RAG:
", result["answer"])


## Section 9: Run the Local Chat App

The notebook teaches the concepts. The chat app gives you a simple UI.

From the repo root, run:

```bash
streamlit run app.py
```

Then ask questions in the browser. Everything stays on your laptop.

**Workshop stretch task:** Point the pipeline at your own document and test 3 questions from your field.
